In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedShuffleSplit, cross_validate
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score,
    f1_score
)
from imblearn.over_sampling import RandomOverSampler

In [2]:
logreg = LogisticRegression(max_iter=5000)

metric_list = ["precision", "recall", "roc_auc", "accuracy", "f1"]

seed = 42
n_splits = 5

# Use stratified splits
kf = StratifiedShuffleSplit(
    n_splits=n_splits,
    test_size=1 / n_splits,
    random_state=seed
)

def evaluate_logistic_regression(model_number, upsample=False):
    """
    Evaluate Logistic Regression using cross-validation and an independent test set.
    """

    # Load training and test data
    X_train_full = pd.read_csv(
        f"../data/X_train_{model_number}.csv",
        keep_default_na=False
    )

    y_train_full = pd.read_csv(
        f"../data/y_train_{model_number}.csv",
        keep_default_na=False
    ).values.ravel()

    X_test = pd.read_csv(
        f"../data/X_test_{model_number}.csv",
        keep_default_na=False
    )

    y_test = pd.read_csv(
        f"../data/y_test_{model_number}.csv",
        keep_default_na=False
    ).values.ravel()

    # Check
    print("\n")
    print(model_number)

    print("X_train shape:", X_train_full.shape)
    print("X_test shape:", X_test.shape)

    print("\ny_train distribution:")
    print(pd.Series(y_train_full).value_counts())

    print("\ny_test distribution:")
    print(pd.Series(y_test).value_counts())


    # Cross-validation
    if upsample:

        # Store scores
        cv_scores = {
            "fold": [],
            "cv_precision": [],
            "cv_recall": [],
            "cv_roc_auc": [],
            "cv_accuracy": [],
            "cv_f1": []
        }

        splits = list(kf.split(X_train_full, y_train_full))

        for fold, (train_idx, val_idx) in enumerate(splits, start=1):

            X_train = X_train_full.iloc[train_idx]
            y_train = y_train_full[train_idx]

            X_val = X_train_full.iloc[val_idx]
            y_val = y_train_full[val_idx]

            # Use upsampling to the training part
            upsampler = RandomOverSampler(random_state=seed)

            X_train_upsample, y_train_upsample = upsampler.fit_resample(
                X_train,
                y_train
            )

            # Train LR on upsampled training data
            clf = LogisticRegression(max_iter=5000)
            clf.fit(X_train_upsample, y_train_upsample)

            # Predict labels and probabilities
            preds = clf.predict(X_val)
            prob_preds = clf.predict_proba(X_val)[:, 1]

            # Save scores
            cv_scores["fold"].append(fold)
            cv_scores["cv_precision"].append(precision_score(y_val, preds, zero_division=0))
            cv_scores["cv_recall"].append(recall_score(y_val, preds, zero_division=0))
            cv_scores["cv_roc_auc"].append(roc_auc_score(y_val, prob_preds))
            cv_scores["cv_accuracy"].append(accuracy_score(y_val, preds))
            cv_scores["cv_f1"].append(f1_score(y_val, preds, zero_division=0))

        cv_results = pd.DataFrame(cv_scores)

    else:

        cv_raw = cross_validate(
            LogisticRegression(max_iter=5000),
            X_train_full,
            y_train_full,
            cv=kf,
            scoring=metric_list,
            return_train_score=False
        )

        cv_results = pd.DataFrame({
            "fold": range(1, n_splits + 1),
            "cv_precision": cv_raw["test_precision"],
            "cv_recall": cv_raw["test_recall"],
            "cv_roc_auc": cv_raw["test_roc_auc"],
            "cv_accuracy": cv_raw["test_accuracy"],
            "cv_f1": cv_raw["test_f1"]
        })

    # Calculate mean and standard error
    cv_summary = cv_results.drop(columns=["fold"]).agg(["mean", "sem"])
    print("\nCross-validation results:")
    display(cv_results)
    print("\nCross-validation summary:")
    display(cv_summary)

    # Test evaluation
    if upsample:

        upsampler = RandomOverSampler(random_state=seed)

        X_train_final, y_train_final = upsampler.fit_resample(
            X_train_full,
            y_train_full
        )

    else:

        X_train_final = X_train_full
        y_train_final = y_train_full

    # Train the final model
    final_model = LogisticRegression(max_iter=5000)

    final_model.fit(X_train_final, y_train_final)

    # Evaluate on test set
    test_preds = final_model.predict(X_test)
    test_prob_preds = final_model.predict_proba(X_test)[:, 1]
    test_results = pd.DataFrame([{
        "test_precision": precision_score(y_test, test_preds, zero_division=0),
        "test_recall": recall_score(y_test, test_preds, zero_division=0),
        "test_roc_auc": roc_auc_score(y_test, test_prob_preds),
        "test_accuracy": accuracy_score(y_test, test_preds),
        "test_f1": f1_score(y_test, test_preds, zero_division=0)
    }])

    print("\nFinal test results:")
    display(test_results)

    print("Mean CV recall:", round(cv_results["cv_recall"].mean(), 3))
    print("Mean CV ROC AUC:", round(cv_results["cv_roc_auc"].mean(), 3))
    print("Mean CV accuracy:", round(cv_results["cv_accuracy"].mean(), 3))

    return cv_results, cv_summary, test_results, final_model

In [3]:
# Run models
model_settings = {
    "model_1": False,
    "model_1a": True,
    "model_1b": True,
    "model_2": False,
    "model_2a": True,
    "model_2b": True
}

cv_results_dict = {}
cv_summary_dict = {}
test_results_list = []
final_model_dict = {}

for model_name, use_upsample in model_settings.items():
    
    (
        cv_results,
        cv_summary,
        test_results,
        final_model
    ) = evaluate_logistic_regression(
        model_name,
        upsample=use_upsample
    )

    cv_results_dict[model_name] = cv_results
    cv_summary_dict[model_name] = cv_summary

    test_results["model"] = model_name
    test_results_list.append(test_results)



model_1
X_train shape: (32108, 59)
X_test shape: (8028, 59)

y_train distribution:
1    17321
0    14787
Name: count, dtype: int64

y_test distribution:
1    4337
0    3691
Name: count, dtype: int64

Cross-validation results:


,fold,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
0,1,0.795620,0.818129,0.866036,0.788539,0.806718
1,2,0.803143,0.811490,0.870778,0.791031,0.807295
2,3,0.797419,0.820439,0.871480,0.790719,0.808765
3,4,0.799432,0.812356,0.868827,0.788851,0.805842
4,5,0.807911,0.819573,0.874344,0.797571,0.813700



Cross-validation summary:


,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
mean,0.800705,0.816397,0.870293,0.791342,0.808464
sem,0.002193,0.001869,0.001384,0.001633,0.001393



Final test results:


...


Cross-validation summary:


,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
mean,0.864393,0.737919,0.787453,0.726123,0.796155
sem,0.002164,0.004157,0.003388,0.004012,0.003287



Final test results:


,test_precision,test_recall,test_roc_auc,test_accuracy,test_f1
0,0.856031,0.727273,0.774186,0.71542,0.786416


Mean CV recall: 0.738
Mean CV ROC AUC: 0.787
Mean CV accuracy: 0.726


In [4]:
# Results
cv_results_all = pd.concat(cv_results_dict, names=["model", "row"])
cv_summary_all = pd.concat(cv_summary_dict, names=["model", "stat"])

test_summary_all = pd.concat(test_results_list, ignore_index=True)
test_summary_all = test_summary_all[
    [
        "model",
        "test_precision",
        "test_recall",
        "test_roc_auc",
        "test_accuracy",
        "test_f1"
    ]
]

print("CV summaries:")
display(cv_summary_all)
print("\nTest results:")
display(test_summary_all)

# Save
cv_results_all.to_csv("../data/logistic_regression_cv_results_all.csv")
cv_summary_all.to_csv("../data/logistic_regression_cv_summary_all.csv")
test_summary_all.to_csv("../data/logistic_regression_test_summary_all.csv", index=False)
with open("../data/logistic_regression_final_models.pkl", "wb") as f:
    pickle.dump(final_model_dict, f)

CV summaries:


cv_precision  cv_recall  cv_roc_auc  cv_accuracy     cv_f1
model    stat                                                            
model_1  mean      0.800705   0.816397    0.870293     0.791342  0.808464
         sem       0.002193   0.001869    0.001384     0.001633  0.001393
model_1a mean      0.484873   0.762338    0.820056     0.730184  0.592718
         sem       0.002011   0.002092    0.001280     0.001696  0.001106
model_1b mean      0.877351   0.777115    0.834692     0.765666  0.824163
         sem       0.001567   0.003916    0.001873     0.001804  0.001796
model_2  mean      0.756431   0.865568    0.781789     0.730769  0.807324
         sem       0.000726   0.002016    0.002068     0.001372  0.001126
model_2a mean      0.681095   0.684335    0.724658     0.663880  0.682706
         sem       0.003734   0.004510    0.005979     0.004111  0.004030
model_2b mean      0.864393   0.737919    0.787453     0.726123  0.796155
         sem       0.002164   0.004157    0.003388     0.004012  0.003287


Test results:


,model,test_precision,test_recall,test_roc_auc,test_accuracy,test_f1
0,model_1,0.806136,0.817846,0.867278,0.795341,0.811949
1,model_1a,0.484848,0.758003,0.820267,0.726330,0.591409
2,model_1b,0.876523,0.768560,0.828338,0.760270,0.818999
3,model_2,0.747614,0.869832,0.768426,0.724215,0.804105
4,model_2a,0.693009,0.715182,0.731749,0.679157,0.703921
5,model_2b,0.856031,0.727273,0.774186,0.715420,0.786416
